In [1]:
from dataclasses import dataclass
from typing import List, Tuple, Optional, Callable, Dict
import unicodedata
import re

def normalize_text(s: str, lowercase: bool = True, strip_punct: bool = False, collapse_spaces: bool = True, remove_diacritics: bool = False) -> str:
    if lowercase:
        s = s.lower()
    if remove_diacritics:
        s = ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))
    if strip_punct:
        s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    if collapse_spaces:
        s = re.sub(r"\s+", " ", s, flags=re.UNICODE).strip()
    return s

def word_tokenize(s: str) -> List[str]:
    s = s.strip()
    return [] if not s else s.split()

@dataclass
class AlignmentStep:
    ref: Optional[str]
    hyp: Optional[str]
    op: str

@dataclass
class EditStats:
    substitutions: int
    deletions: int
    insertions: int
    correct: int

    @property
    def errors(self) -> int:
        return self.substitutions + self.deletions + self.insertions

def align_and_score(ref: List[str], hyp: List[str]) -> Tuple[int, List[AlignmentStep], EditStats]:
    n, m = len(ref), len(hyp)
    dp = [[(0, None) for _ in range(m+1)] for __ in range(n+1)]

    for i in range(1, n+1):
        dp[i][0] = (i, 'D')
    for j in range(1, m+1):
        dp[0][j] = (j, 'I')

    for i in range(1, n+1):
        for j in range(1, m+1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = (dp[i-1][j-1][0], 'C')
            else:
                s = dp[i-1][j-1][0] + 1
                i_cost = dp[i][j-1][0] + 1
                d = dp[i-1][j][0] + 1
                if s <= i_cost and s <= d:
                    dp[i][j] = (s, 'S')
                elif i_cost <= d:
                    dp[i][j] = (i_cost, 'I')
                else:
                    dp[i][j] = (d, 'D')

    i, j = n, m
    alignment: List[AlignmentStep] = []
    stats = EditStats(substitutions=0, deletions=0, insertions=0, correct=0)

    while i > 0 or j > 0:
        op = dp[i][j][1]
        if op == 'C':
            alignment.append(AlignmentStep(ref[i-1], hyp[j-1], 'C'))
            stats.correct += 1
            i, j = i-1, j-1
        elif op == 'S':
            alignment.append(AlignmentStep(ref[i-1], hyp[j-1], 'S'))
            stats.substitutions += 1
            i, j = i-1, j-1
        elif op == 'I':
            alignment.append(AlignmentStep(None, hyp[j-1], 'I'))
            stats.insertions += 1
            j = j-1
        elif op == 'D':
            alignment.append(AlignmentStep(ref[i-1], None, 'D'))
            stats.deletions += 1
            i = i-1

    alignment.reverse()
    return dp[n][m][0], alignment, stats

def format_alignment(alignment: List[AlignmentStep]) -> str:
    ref_line, hyp_line, op_line = [], [], []
    for step in alignment:
        ref_line.append(step.ref if step.ref is not None else "∅")
        hyp_line.append(step.hyp if step.hyp is not None else "∅")
        op_line.append(step.op)
    return "REF: " + " | ".join(ref_line) + "\n" + "HYP: " + " | ".join(hyp_line) + "\n" + "OPS: " + " | ".join(op_line)

def compute_wer(ref_text: str, hyp_text: str, normalizer: Callable[..., str] = normalize_text, **norm_kwargs) -> Dict[str, float]:
    ref_norm = normalizer(ref_text, **norm_kwargs)
    hyp_norm = normalizer(hyp_text, **norm_kwargs)
    ref_tokens = word_tokenize(ref_norm)
    hyp_tokens = word_tokenize(hyp_norm)
    dist, alignment, stats = align_and_score(ref_tokens, hyp_tokens)
    N = max(1, len(ref_tokens))
    wer = stats.errors / N
    return {
        "WER": wer,
        "S": stats.substitutions,
        "D": stats.deletions,
        "I": stats.insertions,
        "C": stats.correct,
        "N_ref_words": len(ref_tokens),
        "alignment": format_alignment(alignment)
    }

if __name__ == "__main__":
    ref = "the quick brown fox"
    hyp = "the quick fox"
    res = compute_wer(ref, hyp, strip_punct=False)
    print(res["alignment"])
    print({k: v for k, v in res.items() if k != "alignment"})

REF: the | quick | brown | fox
HYP: the | quick | ∅ | fox
OPS: C | C | D | C
{'WER': 0.25, 'S': 0, 'D': 1, 'I': 0, 'C': 3, 'N_ref_words': 4}
